# Analisi del dataset

Dataset **KAt-DataCenter** dell'Università di Paderborn (Lessmeier et al., 2016), usato da
Toma, Piltan e Kim (2021) per la diagnosi di guasto ai cuscinetti dalla corrente di statore.

Il criterio seguito è uno solo: **non assumere nulla dalla documentazione, misurare tutto sui file.**
Le grandezze dichiarate dal banco prova (frequenza di campionamento, durata, velocità, numero di
coppie polari) vengono ricavate dai dati e confrontate con i valori attesi.

Il notebook produce, nella cartella `risultati/01_analisi_dataset/`:

* `tabelle/` — i risultati numerici in CSV
* `figure/` — le figure usate nella relazione

Funzioni condivise in `codice/funzioni.py`, costanti e anagrafica in `codice/config.py`.

## Preparazione dell'ambiente

In [ ]:
!apt-get -qq update && apt-get -qq install -y unrar
!pip -q install requests scipy pymupdf

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/bearings_detection/codice')

import os, time, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
import funzioni as fn

P = config.percorsi(sottocartella='01_analisi_dataset')
fn.stile_grafici()

print('cartelle del progetto:')
for chiave, valore in P.items():
    print(f'  {chiave:12s} {valore}')

## 1. Cosa contiene il repository

Prima di scaricare qualunque cosa, guardiamo che cosa il repository mette a disposizione.

In [ ]:
cuscinetti_remoti = fn.elenco_remoto()

tipi = {}
for nome in cuscinetti_remoti:
    prefisso = nome.rstrip('0123456789')
    tipi[prefisso] = tipi.get(prefisso, 0) + 1

print(len(cuscinetti_remoti), 'cuscinetti disponibili')
print('per tipo:', tipi)
print()
print('  K  = sani')
print('  KA = danno sulla pista esterna')
print('  KI = danno sulla pista interna')
print('  KB = danno combinato')
print()

import requests
print('--- readme del repository ---')
print(requests.get(config.URL_REPOSITORY + 'readme_versions.txt').text)

In [ ]:
# Scarica gli archivi mancanti. Rieseguendo il notebook questa cella non fa nulla.
archivi, dimensione = fn.scarica_archivi(cuscinetti_remoti, P['raw'])
print(len(archivi), 'archivi su Drive |', round(dimensione / 1e9, 2), 'GB')

In [ ]:
# Ogni archivio contiene due PDF: la scheda del danno e il measuring log.
# Sono la fonte dell'anagrafica usata nella Sezione 7.
schede = fn.estrai_schede(cuscinetti_remoti, P['raw'], P['documenti'])
for nome in sorted(schede)[:3]:
    print(nome, '->', schede[nome])
print('...')
print(len(schede), 'cuscinetti con schede estratte')

## 2. Struttura di un file di misura

Ogni registrazione è un file MATLAB con quattro campi: `Info`, `X`, `Y`, `Description`.
In `Y` stanno i segnali acquisiti, in `X` gli assi dei tempi. Il punto importante è che
**i segnali non condividono la stessa frequenza di campionamento**: ciascuno dichiara a
quale asse fa riferimento. Leggerli assumendo un'unica frequenza porta a segmentazioni sbagliate.

In [ ]:
# Estrae i 17 cuscinetti della Tabella 2 del paper. Alcuni minuti la prima volta.
fn.estrai_misure(config.CUSCINETTI_PAPER, P['raw'], P['estratti'])

esempio = fn.elenco_registrazioni(['K001'], P['estratti'], [config.REGIME_PRINCIPALE])[0]
struct = fn.apri(esempio)

print('file di esempio:', os.path.basename(esempio))
print('campi della struttura:', list(struct.keys()))
print()
print('segnali in Y :', [s['Name'] for s in struct['Y']])
print('assi in X    :', [x['Raster'] for x in struct['X']])

In [ ]:
# Gli assi dei tempi contengono istanti, quindi il passo di campionamento si misura.
# Lo ricaviamo dall'intervallo complessivo diviso il numero di passi: sul singolo passo
# pesa l'arrotondamento della memorizzazione, sull'intero intervallo no.
assi = fn.assi_dei_tempi(esempio, struct)
assi

In [ ]:
# Verifica sul canale che useremo. Se frequenza o durata si discostano da quelle attese
# oltre la tolleranza la funzione solleva un'eccezione: meglio fermarsi che proseguire
# con una lunghezza di frame sbagliata.
misura = fn.misura_fs(esempio, config.CANALE_CORRENTE, struct)

print(f"canale        : {config.CANALE_CORRENTE}")
print(f"raster        : {misura['raster']}")
print(f"campioni      : {misura['campioni']}")
print(f"frequenza     : {misura['frequenza_Hz']:.4f} Hz   (attesa {config.FS_ATTESO})")
print(f"durata        : {misura['durata_s']:.6f} s    (attesa {config.DURATA_ATTESA})")
print()
print(f"lunghezza del frame a {config.REGIMI[config.REGIME_PRINCIPALE]['rpm']} rpm:",
      fn.lunghezza_frame(config.REGIMI[config.REGIME_PRINCIPALE]['rpm']), 'campioni')

In [ ]:
tabella_canali = fn.canali(esempio, struct)
fn.salva_tabella(tabella_canali, 'canali_disponibili', P['tabelle'])
tabella_canali

## 3. Che aspetto hanno i segnali

Confronto visivo fra le tre classi, nello stesso regime operativo.

In [ ]:
esempi = [('K001', 'normale'), ('KA04', 'guasto pista esterna'), ('KI04', 'guasto pista interna')]

fig, assi_fig = plt.subplots(len(esempi), 1, figsize=(9.5, 6.5), sharex=True, sharey=True)
for ax, (cuscinetto, descrizione) in zip(assi_fig, esempi):
    percorso = fn.elenco_registrazioni([cuscinetto], P['estratti'], [config.REGIME_PRINCIPALE])[0]
    x = fn.leggi(percorso, config.CANALE_CORRENTE)
    ax.plot(np.arange(3000) / config.FS_ATTESO * 1000, x[:3000],
            linewidth=0.7, color=fn.COLORI['scuro'])
    ax.set_title(f'{descrizione}  ({cuscinetto})', fontsize=9.5, loc='left')
    ax.set_ylabel('corrente (A)')
assi_fig[-1].set_xlabel('tempo (ms)')
fig.suptitle(f'Corrente di fase, regime {config.REGIME_PRINCIPALE}', fontsize=10.5)

fn.salva_figura(fig, 'forme_onda_per_classe', P['figure'])
plt.show()

## 4. Scansione completa e controlli di qualità

Una sola passata su tutte le registrazioni dei 17 cuscinetti nei quattro regimi.
Per ogni file misuriamo frequenza e durata, calcoliamo le statistiche del segnale di
corrente e della velocità, e memorizziamo un'impronta del segnale per la ricerca di
duplicati della Sezione 6. Tutto il resto del notebook lavora su questa tabella.

In [ ]:
percorsi_mat = fn.elenco_registrazioni(config.CUSCINETTI_PAPER, P['estratti'])
print(len(percorsi_mat), 'registrazioni da esaminare')

righe, inizio = [], time.time()
for k, percorso in enumerate(percorsi_mat, 1):
    riga = fn.metadati_nome(percorso)
    riga['classe'] = config.CLASSE_DI[riga['cuscinetto']]

    struct = fn.apri(percorso)
    corrente = fn.leggi(percorso, config.CANALE_CORRENTE, struct)
    velocita = fn.leggi(percorso, 'speed', struct)

    try:
        misura = fn.misura_fs(percorso, config.CANALE_CORRENTE, struct)
        riga['fs_Hz'] = misura['frequenza_Hz']
        riga['durata_s'] = misura['durata_s']
        riga['fuori_tolleranza'] = False
    except ValueError:
        riga['fs_Hz'] = np.nan
        riga['durata_s'] = np.nan
        riga['fuori_tolleranza'] = True

    riga.update(fn.caratteristiche(corrente))

    vel = fn.stazionarieta(velocita)
    riga['rpm_medio'] = vel['media']
    riga['rpm_min'] = vel['minimo']
    riga['rpm_max'] = vel['massimo']
    riga['rpm_variazione_pct'] = vel['variazione_pct']

    riga['firma'] = hashlib.blake2b(np.ascontiguousarray(corrente).tobytes(),
                                    digest_size=8).hexdigest()
    righe.append(riga)

    if k % 200 == 0:
        print(f'  {k}/{len(percorsi_mat)}   ({time.time() - inizio:.0f} s)')

scan = pd.DataFrame(righe)
print(f'\nscansione completata in {time.time() - inizio:.0f} s ->', scan.shape)
fn.salva_tabella(scan.drop(columns=['file']), 'scansione_completa', P['tabelle'])

In [ ]:
# L'inventario verifica che l'estrazione sia completa: un'estrazione parziale non
# produce errori, produce silenziosamente meno dati.
riepilogo = (scan.groupby(['regime', 'classe'])
                 .agg(registrazioni=('registrazione', 'count'),
                      cuscinetti=('cuscinetto', 'nunique'))
                 .reset_index())
riepilogo['classe_nome'] = riepilogo['classe'].map(dict(enumerate(config.NOMI_CLASSI)))

atteso = len(config.CUSCINETTI_PAPER) * 20
print('registrazioni attese per regime:', atteso, '(17 cuscinetti x 20 ripetizioni)')
for regime, gruppo in scan.groupby('regime'):
    print(f'  {regime}: {len(gruppo)}', '  OK' if len(gruppo) == atteso else '  <-- INCOMPLETO')

fn.salva_tabella(riepilogo, 'inventario_per_regime_classe', P['tabelle'])
riepilogo

In [ ]:
print('--- controlli di integrità ---')
print('registrazioni totali          :', len(scan))
print('valori non finiti             :', int((scan['non_finiti'] > 0).sum()))
print('canali piatti (std = 0)       :', int((scan['std'] == 0).sum()))
print('durata fuori tolleranza       :', int(scan['fuori_tolleranza'].sum()))
print()
print('--- frequenza di campionamento misurata ---')
print(scan['fs_Hz'].describe()[['min', 'mean', 'max']].to_string())
print()
print('--- fattore di cresta ---')
print(scan['crest'].describe()[['min', '50%', 'max']].to_string())
print()
print('registrazioni con fattore di cresta più alto:')
scan.nlargest(5, 'crest')[['registrazione', 'cuscinetto', 'regime', 'crest',
                           'rpm_variazione_pct']]

In [ ]:
# Le statistiche nel dominio del tempo separano le classi?
per_classe = scan.groupby('classe')[['std', 'picco', 'rms', 'crest', 'zcr', 'f_dominante']].median()
per_classe.index = [config.NOMI_CLASSI[i] for i in per_classe.index]
print(per_classe.to_string())

grandezze = ['rms', 'picco', 'std', 'crest']
fig, assi_fig = plt.subplots(1, len(grandezze), figsize=(11, 3.2))
for ax, g in zip(assi_fig, grandezze):
    valori = [scan.loc[scan['classe'] == c, g] for c in range(3)]
    bp = ax.boxplot(valori, patch_artist=True, widths=0.6,
                    medianprops=dict(color=fn.COLORI['scuro']), showfliers=False)
    ax.set_xticks(range(1, len(config.NOMI_CLASSI) + 1))
    ax.set_xticklabels(config.NOMI_CLASSI)
    for corpo, colore in zip(bp['boxes'], [fn.COLORI['normale'], fn.COLORI['esterno'],
                                           fn.COLORI['interno']]):
        corpo.set_facecolor(colore)
    ax.set_title(g, fontsize=10)
    ax.tick_params(axis='x', labelsize=8.5)
fig.suptitle('Statistiche nel dominio del tempo: le tre classi sono sovrapposte', fontsize=10.5)

fn.salva_figura(fig, 'statistiche_per_classe', P['figure'])
fn.salva_tabella(per_classe.reset_index().rename(columns={'index': 'classe'}),
                 'statistiche_mediane_per_classe', P['tabelle'])
plt.show()

In [ ]:
# La registrazione più anomala dell'intero dataset.
peggiore = scan.nlargest(1, 'crest').iloc[0]
print(peggiore['registrazione'], '| fattore di cresta',
      round(peggiore['crest'], 2), '| variazione di velocità',
      round(peggiore['rpm_variazione_pct'], 2), '%')

x = fn.leggi(peggiore['file'], config.CANALE_CORRENTE)
indice = int(np.argmax(np.abs(x)))
inizio_zoom = max(0, indice - 300)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.2))
a1.plot(x, linewidth=0.4, color=fn.COLORI['scuro'])
a1.set_title(peggiore['registrazione'], fontsize=9.5)
a1.set_xlabel('campione'); a1.set_ylabel('corrente (A)')
a2.plot(range(inizio_zoom, indice + 300), x[inizio_zoom:indice + 300],
        linewidth=0.9, color=fn.COLORI['accento'])
a2.set_title('zoom sul picco', fontsize=9.5)
a2.set_xlabel('campione')

fn.salva_figura(fig, 'registrazione_anomala', P['figure'])
plt.show()

## 5. Velocità e tipo di macchina elettrica

Due verifiche collegate. La prima è che la velocità sia stabile dentro ogni registrazione,
perché la segmentazione a giro fisso lo presuppone. La seconda risponde a una domanda più
importante: **il motore del banco è sincrono o asincrono?**

L'articolo di Toma et al. presenta il lavoro come diagnosi su motori a induzione, ma nella
descrizione del banco riporta un motore sincrono a magneti permanenti. La distinzione si
verifica sui dati. Per una macchina sincrona vale

$$\frac{f_{se}}{f_m} = p$$

esattamente, e **indipendentemente dal carico**. Per una macchina asincrona lo stesso rapporto
vale $p/(1-s)$ e quindi **varia con il carico**, perché lo scorrimento cresce con la coppia
resistente. Basta calcolare il rapporto nei quattro regimi e vedere se resta costante.

In [ ]:
velocita_regime = scan.groupby('regime')[['rpm_medio', 'rpm_variazione_pct']].agg(['mean', 'max'])
print('--- velocità per regime ---')
print(velocita_regime.to_string())
print()

scan['rpm_nominale'] = scan['regime'].map(lambda r: config.REGIMI[r]['rpm'])
scan['scarto_pct'] = 100 * (scan['rpm_medio'] - scan['rpm_nominale']) / scan['rpm_nominale']
print('--- nominale contro misurata ---')
print(scan.groupby('regime')[['rpm_nominale', 'rpm_medio', 'scarto_pct']].mean().to_string())

In [ ]:
scan['giri_al_secondo'] = scan['rpm_medio'] / 60.0
scan['rapporto'] = scan['f_dominante'] / scan['giri_al_secondo']

coppie = scan.groupby('regime')[['giri_al_secondo', 'f_dominante', 'rapporto']].mean()
coppie['coppia_Nm'] = [config.REGIMI[r]['coppia_Nm'] for r in coppie.index]
coppie['forza_N'] = [config.REGIMI[r]['forza_N'] for r in coppie.index]
print(coppie.to_string())
print()
print('scarto massimo dal valore intero:',
      f"{(coppie['rapporto'] - round(coppie['rapporto'].mean())).abs().max():.2e}")

ordine = ['N09_M07_F10', 'N15_M01_F10', 'N15_M07_F04', 'N15_M07_F10']
fig, ax = plt.subplots(figsize=(7.2, 3.8))
ax.plot(range(len(ordine)), coppie.loc[ordine, 'rapporto'], 'o-',
        color=fn.COLORI['scuro'], markersize=8, linewidth=1.6, zorder=3)
ax.axhline(4.0, color=fn.COLORI['accento'], linestyle='--', linewidth=1.3, zorder=2)
ax.text(len(ordine) - 1, 4.0 + 0.0004, 'p = 4 atteso per macchina sincrona',
        color=fn.COLORI['accento'], ha='right', va='bottom', fontsize=8.5)
etichette = [f"{r}\n{config.REGIMI[r]['rpm']} rpm\n{config.REGIMI[r]['coppia_Nm']} Nm\n"
             f"{config.REGIMI[r]['forza_N']} N" for r in ordine]
ax.set_xticks(range(len(ordine)))
ax.set_xticklabels(etichette, fontsize=7.5)
ax.set_ylabel('$f_{dominante}$ / $f_{rotazione}$')
ax.set_title('Il rapporto non dipende dal carico: la macchina è sincrona')

fn.salva_figura(fig, 'coppie_polari', P['figure'])
fn.salva_tabella(coppie.reset_index(), 'rapporto_coppie_polari', P['tabelle'])
plt.show()

## 6. Registrazioni duplicate

Il measuring log di KA04 documenta che un file conteneva dati errati e fu sostituito con
una copia di un altro. Verifichiamo in modo indipendente, confrontando l'impronta dei
segnali, se esistano registrazioni identiche — documentate o meno.

La cosa ha una conseguenza pratica: con una suddivisione casuale dei segmenti, due
registrazioni identiche finiscono garantitamente sia in addestramento sia in verifica.

In [ ]:
doppie = scan[scan.duplicated('firma', keep=False)]

print('registrazioni con contenuto identico a un\'altra:', len(doppie))
if len(doppie):
    print('cuscinetti coinvolti:', sorted(doppie['cuscinetto'].unique()))
    print()
    print(doppie.sort_values(['cuscinetto', 'regime', 'firma'])
               [['cuscinetto', 'regime', 'registrazione', 'firma']].to_string(index=False))
    fn.salva_tabella(doppie[['cuscinetto', 'regime', 'registrazione', 'firma']],
                     'registrazioni_duplicate', P['tabelle'])

## 7. La natura dei singoli cuscinetti

Le schede allegate agli archivi descrivono ciascun esemplare: rodaggio, meccanismo di
danneggiamento, componente interessato, estensione e costruttore. Sono informazioni che
non si ricavano dai segnali, e mostrano che **le tre classi non sono insiemi omogenei**.

In [ ]:
anagrafica = fn.anagrafica_cuscinetti()
fn.salva_tabella(anagrafica, 'anagrafica_cuscinetti', P['tabelle'])

print('--- cuscinetti sani: ore di funzionamento precedenti alle misure ---')
sani = anagrafica[anagrafica['classe'] == 0].sort_values('ore')
print(sani[['cuscinetto', 'ore', 'produttore']].to_string(index=False))
print(f"\nrapporto fra il massimo e il minimo: {sani['ore'].max() / sani['ore'].min():.0f}x")

print('\n--- guasti: meccanismo, estensione, danno secondario ---')
guasti = anagrafica[anagrafica['classe'] > 0]
print(guasti[['cuscinetto', 'classe_nome', 'modo', 'caratteristica',
              'estensione', 'secondario']].to_string(index=False))

print('\n--- classe contro costruttore ---')
print(pd.crosstab(anagrafica['classe_nome'], anagrafica['produttore']).to_string())
solo_ibu = anagrafica['produttore'].str.startswith('IBU')
concordi = int(((anagrafica['classe'] == 0) == solo_ibu).sum())
print(f"\nla sola regola 'costruttore IBU => sano' classifica correttamente "
      f"{concordi} cuscinetti su {len(anagrafica)}")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.9))

# rodaggio dei sani
barre = a1.bar(sani['cuscinetto'], sani['ore'], width=0.62,
               color=[fn.COLORI['normale']] * (len(sani) - 1) + [fn.COLORI['accento']])
for r, v in zip(barre, sani['ore']):
    a1.annotate('> 50' if v >= 50 else f'{v:.0f}',
                (r.get_x() + r.get_width() / 2, v), textcoords='offset points',
                xytext=(0, 3), ha='center', fontsize=8.5)
a1.set_ylabel('ore di funzionamento')
a1.set_title('Cuscinetti sani: rodaggio dichiarato')
a1.set_ylim(0, sani['ore'].max() * 1.2)

# estensione del danno per classe
for classe, colore in [(1, fn.COLORI['esterno']), (2, fn.COLORI['interno'])]:
    parte = anagrafica[anagrafica['classe'] == classe].sort_values('cuscinetto')
    a2.bar(parte['cuscinetto'], parte['estensione'], width=0.62, color=colore,
           label=config.NOMI_CLASSI[classe])
a2.set_ylabel('estensione del danno (scala 1-3)')
a2.set_title('Gravità del danno dentro la stessa etichetta')
a2.tick_params(axis='x', labelsize=8, rotation=45)
a2.legend(fontsize=8.5)

fn.salva_figura(fig, 'natura_cuscinetti', P['figure'])
plt.show()

In [ ]:
# Il diametro primitivo non è uguale per tutti: dipende dal costruttore, e sposta
# le frequenze caratteristiche attese.
rpm = config.REGIMI[config.REGIME_PRINCIPALE]['rpm']
righe = []
for D in sorted(set(config.DIAMETRO_PRIMITIVO_MM.values())):
    f = fn.frequenze_guasto(rpm / 60.0, D=D)
    quali = sorted(p for p, v in config.DIAMETRO_PRIMITIVO_MM.items() if v == D)
    righe.append({'diametro_primitivo_mm': D, 'costruttori': ', '.join(quali),
                  'BPFO_Hz': f['BPFO'], 'BPFI_Hz': f['BPFI'],
                  'BSF_Hz': f['BSF'], 'FTF_Hz': f['FTF']})

frequenze = pd.DataFrame(righe)
print(f'frequenze caratteristiche a {rpm} rpm:')
print(frequenze.round(2).to_string(index=False))
print(f"\nscarto su BPFO: {abs(frequenze['BPFO_Hz'].diff().iloc[-1]):.2f} Hz")

fn.salva_tabella(frequenze, 'frequenze_caratteristiche', P['tabelle'])

## 8. Riepilogo

In [ ]:
print('FIGURE prodotte')
for f in sorted(os.listdir(P['figure'])):
    print('  ', f)

print()
print('TABELLE prodotte')
for f in sorted(os.listdir(P['tabelle'])):
    print('  ', f)

print()
print('SINTESI')
print('  registrazioni esaminate  ', len(scan))
print('  cuscinetti               ', scan['cuscinetto'].nunique())
print('  regimi operativi         ', scan['regime'].nunique())
print('  frequenza misurata       ', round(scan['fs_Hz'].mean(), 4), 'Hz  (attesa',
      config.FS_ATTESO, ')')
print('  anomalie di integrita    ', int(scan['fuori_tolleranza'].sum()))
print('  registrazioni duplicate  ', len(doppie))
print('  rapporto f_el / f_rot    ', round(scan['rapporto'].mean(), 4),
      '-> p =', round(scan['rapporto'].mean()))